# Sprint 4: DARPA / IARPA Biosecurity Programs

**Goal:** Map US government biosecurity programs to their performers. Show which workshop-affiliated labs are government contractors.

**Data Sources:**
1. **USASpending.gov API** — Federal contract/grant data with amounts, dates, recipient names
2. **DARPA program pages** — HTML snapshots for performer attribution

**Programs of interest:**
- P3 (Pandemic Prevention Platform) — DARPA
- PREEMPT (PREventing EMerging Pathogenic Threats) — DARPA
- SAFE GENES — DARPA
- PREPARE (PREemptive Expression of Protective Alleles and Response Elements) — DARPA
- Fun GCAT (Functional Genomic and Computational Assessment of Threats) — IARPA

## Step 1: Load Raw Data from USASpending.gov API

We queried USASpending.gov with the following strategy:
- **Agency filter**: Defense Advanced Research Projects Agency (subtier)
- **Keywords**: biological, pandemic, pathogen, biosecurity, antibody, countermeasure, gene editing, genomic, zoonotic, biodefense, biothreats, virus, infectious
- **Time range**: 2010–2025
- **Award types**: Contracts (A/B/C/D) and Grants (02/03/04/05) in separate queries (API limitation)

Raw JSON responses saved in `data/raw/sprint4/`.

In [1]:
import json
import os
import re
from collections import Counter, defaultdict

BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
RAW_DIR = os.path.join(BASE, 'data', 'raw', 'sprint4')
GRAPH_FILE = os.path.join(BASE, 'data', 'graph_data.json')

# Load the two main DARPA BTO queries
with open(os.path.join(RAW_DIR, 'darpa_bto_all_contracts.json')) as f:
    contracts_data = json.load(f)
with open(os.path.join(RAW_DIR, 'darpa_bto_all_grants.json')) as f:
    grants_data = json.load(f)

contracts = contracts_data.get('results', [])
grants = grants_data.get('results', [])

print(f'DARPA BTO contracts: {len(contracts)} awards')
print(f'DARPA BTO grants:    {len(grants)} awards')
print(f'Total awards:        {len(contracts) + len(grants)}')

DARPA BTO contracts: 59 awards
DARPA BTO grants:    51 awards
Total awards:        110


## Step 2: Explore Contract Data

In [2]:
# Show all DARPA BTO contracts sorted by amount
print(f"{'Amount':>15}  {'Recipient':<55} Description")
print('=' * 140)

total_contracts_amt = 0
for r in sorted(contracts, key=lambda x: x.get('Award Amount', 0) or 0, reverse=True):
    amt = r.get('Award Amount', 0) or 0
    total_contracts_amt += amt
    name = (r.get('Recipient Name') or 'N/A')[:55]
    desc = (r.get('Description') or 'N/A')[:80]
    print(f'${amt:>14,.0f}  {name:<55} {desc}')

print(f'\nTotal contract value: ${total_contracts_amt:,.0f}')

         Amount  Recipient                                               Description
$    35,644,311  LEIDOS, INC.                                            THE GOAL OF THE ADAPTIVE RADAR COUNTERMEASURES (ARC) PROGRAM IS TO DEVELOP THE C
$    33,084,890  BOOZ ALLEN HAMILTON INC                                 BTO BIOMEDICAL ENGINEERING SYSTEMS/MOLECULAR BIOLOGY/BIOSECURITY/MEDICAL SCIENCE
$    19,590,427  PRESIDENT AND FELLOWS OF HARVARD COLLEGE                PROPHECY PROGRAM, ULTRAHIGH THROUGHOUT VIRUS-HOST CELL PICOREACTOR SYSTEM FOR PR
$    16,100,251  SCHAFER GOVERNMENT SERVICES, LLC                        BIOCHEMISTRY & GENOMICS, NANOENGINEERING, CHEMISTRY AND FRONT OFFICE TECHNICAL &
$    13,999,878  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO, PROPHECY PROGRAM. LINKING VIRUS POPULATION GENETIC STRUCTURE TO INFECTIVITY AND 
$    12,234,739  TRUSTEES OF THE UNIVERSITY OF PENNSYLVANIA, THE         IGF::OT::IGF LOW RESOURCES LANGUAGES FOR EMERGENT INCIDENTS (LORELEI)THE

## Step 3: Explore Grant Data

In [3]:
# Show all DARPA BTO grants sorted by amount
print(f"{'Amount':>15}  {'Recipient':<55} Description")
print('=' * 140)

total_grants_amt = 0
for r in sorted(grants, key=lambda x: x.get('Award Amount', 0) or 0, reverse=True):
    amt = r.get('Award Amount', 0) or 0
    total_grants_amt += amt
    name = (r.get('Recipient Name') or 'N/A')[:55]
    desc = (r.get('Description') or 'N/A')[:80]
    print(f'${amt:>14,.0f}  {name:<55} {desc}')

print(f'\nTotal grant value: ${total_grants_amt:,.0f}')

         Amount  Recipient                                               Description
$    36,151,284  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO, THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCE
$    26,975,142  UNIVERSITY OF NORTH CAROLINA AT CHAPEL HILL             THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCE
$    23,343,160  PRESIDENT AND FELLOWS OF HARVARD COLLEGE                THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCE
$    14,294,461  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO, THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCE
$    13,942,324  THE BROAD INSTITUTE, INC                                THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCE
$    12,774,922  YALE UNIV                                               THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE

## Step 4: Filter for Bio-Relevant Awards Only

The USASpending.gov keyword search is OR-based (any keyword match). Some results may be false positives (e.g., LORELEI language program mentions 'virus' in context but isn't biosecurity). We apply a tighter bio-relevance filter.

In [4]:
# Bio-relevance filter for DARPA awards
BIO_REL = re.compile(
    r'biolog|biosecur|biosafety|pandemic|pathogen|genomic|sequenc|'
    r'protein|antiviral|vaccine|virol|antibod|countermeasure|'
    r'gene edit|gene drive|CRISPR|infectious|epidemic|'
    r'zoonotic|biosurveillance|biodefense|biothreats|'
    r'PREEMPT|SAFE GENES|PREPARE|pandemic prevention|'
    r'PROPHECY|SIGMA\+|biological threat|virus|'
    r'diagnostic|bioengineering|molecular biology|'
    r'reef|coral',  # REEFENSE is bio-engineering program
    re.I
)

# False positive patterns to exclude
FALSE_POS = re.compile(
    r'LORELEI|RADAR|ADAPTIVE RADAR|ELECTRONIC WARFARE|'
    r'SATELLITE|TRANSPONDER|OPTICAL|LASER|INFRARED|'
    r'CHEMICAL AGENT DETECTOR|MINE DETECTION',
    re.I
)

def is_bio_relevant(award):
    desc = (award.get('Description') or '')
    name = (award.get('Recipient Name') or '')
    text = f"{desc} {name}"
    
    # Exclude known false positives
    if FALSE_POS.search(desc):
        return False, 'false_positive'
    
    # Check bio-relevance
    match = BIO_REL.search(text)
    if match:
        return True, match.group()
    return False, 'no_match'

# Apply filter
all_awards = contracts + grants
bio_awards = []
excluded = []

for a in all_awards:
    relevant, reason = is_bio_relevant(a)
    if relevant:
        a['_match'] = reason
        bio_awards.append(a)
    else:
        excluded.append((a, reason))

print(f'Total awards from USASpending: {len(all_awards)}')
print(f'Bio-relevant awards:           {len(bio_awards)}')
print(f'Excluded:                      {len(excluded)}')
print(f'\nBio-relevant total: ${sum(a.get("Award Amount", 0) or 0 for a in bio_awards):,.0f}')

print(f'\n--- Excluded awards ---')
for a, reason in excluded:
    amt = a.get('Award Amount', 0) or 0
    print(f'  [{reason}] ${amt:>12,.0f} | {(a.get("Recipient Name") or "")[:40]} | {(a.get("Description") or "")[:80]}')

Total awards from USASpending: 110
Bio-relevant awards:           103
Excluded:                      7

Bio-relevant total: $472,165,720

--- Excluded awards ---
  [false_positive] $  35,644,311 | LEIDOS, INC. | THE GOAL OF THE ADAPTIVE RADAR COUNTERMEASURES (ARC) PROGRAM IS TO DEVELOP THE C
  [false_positive] $  12,234,739 | TRUSTEES OF THE UNIVERSITY OF PENNSYLVAN | IGF::OT::IGF LOW RESOURCES LANGUAGES FOR EMERGENT INCIDENTS (LORELEI)THE GOAL OF
  [false_positive] $  10,531,725 | UNIVERSITY OF SOUTHERN CALIFORNIA | IGF::OT::IGF LOW RESOURCE LANGUAGES FOR EMERGENT INCIDENTS (LORELEI) THE GOAL OF
  [false_positive] $   4,906,661 | VADUM INC | THE GOAL OF THE ADAPTIVE RADAR COUNTERMEASURES (ARC) PROGRAM IS TO DEVELOP THE C
  [false_positive] $   4,654,505 | MICHIGAN TECHNOLOGICAL UNIVERSITY | THE GOAL OF THE ADAPTIVE RADAR COUNTERMEASURES (ARC) PROGRAM IS TO DEVELOP THE C
  [false_positive] $   4,040,197 | SYSTEMS & TECHNOLOGY RESEARCH LLC | THE GOAL OF THE ADAPTIVE RADAR COUNTERMEASURE

## Step 5: Identify DARPA BTO Programs from Award Descriptions

DARPA awards don't have a clean 'program' field. We infer programs from award descriptions.

In [5]:
# Program identification from descriptions
PROGRAM_PATTERNS = {
    'P3 (Pandemic Prevention Platform)': re.compile(r'pandemic prevention|P3 program', re.I),
    'PREEMPT': re.compile(r'PREEMPT|preventing emerging pathogenic', re.I),
    'SAFE GENES': re.compile(r'safe genes|genome editing.*safe', re.I),
    'PREPARE': re.compile(r'PREPARE|protective alleles|response elements', re.I),
    'PROPHECY': re.compile(r'PROPHECY|predictive.*viral.*evolution|virus.*host.*picoreactor', re.I),
    'SIGMA+': re.compile(r'SIGMA\+|SIGMA PLUS|biological threat', re.I),
    'REEFENSE': re.compile(r'REEFENSE|reef.*mimic|coral.*reef', re.I),
    'SD2 (Synergistic Discovery & Design)': re.compile(r'SD2|synergistic discovery', re.I),
    'BTO General Support': re.compile(r'BTO.*(?:support|front office|technical.*admin)', re.I),
    'COVID-19 Response': re.compile(r'COVID.?19', re.I),
}

program_awards = defaultdict(list)
unclassified = []

for a in bio_awards:
    desc = (a.get('Description') or '')
    classified = False
    for prog_name, pattern in PROGRAM_PATTERNS.items():
        if pattern.search(desc):
            program_awards[prog_name].append(a)
            classified = True
            break  # First match wins
    if not classified:
        unclassified.append(a)

print(f"{'Program':<45} {'Awards':>6} {'Total Amount':>15}")
print('=' * 70)
for prog, awards in sorted(program_awards.items(), key=lambda x: sum(a.get('Award Amount', 0) or 0 for a in x[1]), reverse=True):
    total = sum(a.get('Award Amount', 0) or 0 for a in awards)
    print(f'{prog:<45} {len(awards):>6} ${total:>14,.0f}')

if unclassified:
    unclass_total = sum(a.get('Award Amount', 0) or 0 for a in unclassified)
    print(f'{"[Unclassified]":<45} {len(unclassified):>6} ${unclass_total:>14,.0f}')
    print(f'\n--- Unclassified awards ---')
    for a in sorted(unclassified, key=lambda x: x.get('Award Amount', 0) or 0, reverse=True):
        amt = a.get('Award Amount', 0) or 0
        print(f'  ${amt:>12,.0f} | {(a.get("Recipient Name") or "")[:40]} | {(a.get("Description") or "")[:90]}')

Program                                       Awards    Total Amount
PROPHECY                                           4 $    43,355,057
BTO General Support                                2 $    33,570,947
REEFENSE                                           2 $    21,882,661
PREEMPT                                            1 $    11,770,931
SD2 (Synergistic Discovery & Design)               1 $    10,446,304
SIGMA+                                             1 $     8,740,317
COVID-19 Response                                  1 $     1,249,032
[Unclassified]                                    91 $   341,150,472

--- Unclassified awards ---
  $  36,151,284 | REGENTS OF THE UNIVERSITY OF CALIFORNIA, | THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCED RESEARCH
  $  26,975,142 | UNIVERSITY OF NORTH CAROLINA AT CHAPEL H | THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCED RESEARCH
  $  23,343,160 | PRESIDENT AND FELLOWS OF HAR

## Step 6: Analyze Recipients (Performers)

In [6]:
# Aggregate by recipient
recipient_totals = defaultdict(lambda: {'count': 0, 'amount': 0, 'programs': set(), 'award_types': set()})

for a in bio_awards:
    name = (a.get('Recipient Name') or 'Unknown').strip()
    amt = a.get('Award Amount', 0) or 0
    atype = a.get('Award Type', '')
    
    recipient_totals[name]['count'] += 1
    recipient_totals[name]['amount'] += amt
    recipient_totals[name]['award_types'].add(atype)
    
    # Tag program
    desc = (a.get('Description') or '')
    for prog_name, pattern in PROGRAM_PATTERNS.items():
        if pattern.search(desc):
            recipient_totals[name]['programs'].add(prog_name)
            break

print(f"{'Recipient':<55} {'Awards':>6} {'Amount':>15} Programs")
print('=' * 120)
for name, info in sorted(recipient_totals.items(), key=lambda x: x[1]['amount'], reverse=True):
    progs = ', '.join(sorted(info['programs'])) if info['programs'] else '[unclassified]'
    print(f"{name[:55]:<55} {info['count']:>6} ${info['amount']:>14,.0f} {progs}")

print(f'\nTotal unique recipients: {len(recipient_totals)}')
print(f'Total bio-relevant funding: ${sum(v["amount"] for v in recipient_totals.values()):,.0f}')

Recipient                                               Awards          Amount Programs
REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO,      3 $    64,445,623 PROPHECY
PRESIDENT AND FELLOWS OF HARVARD COLLEGE                     5 $    57,210,425 PREEMPT, PROPHECY
BOOZ ALLEN HAMILTON INC                                      1 $    33,084,890 BTO General Support
UNIVERSITY OF NORTH CAROLINA AT CHAPEL HILL                  1 $    26,975,142 [unclassified]
MICHIGAN TECHNOLOGICAL UNIVERSITY                            4 $    17,107,169 [unclassified]
SCHAFER GOVERNMENT SERVICES, LLC                             1 $    16,100,251 [unclassified]
TEXAS A&M ENGINEERING EXPERIMENT STATION                     2 $    15,630,642 [unclassified]
YALE UNIV                                                    3 $    14,323,586 [unclassified]
THE BROAD INSTITUTE, INC                                     1 $    13,942,324 [unclassified]
MASSACHUSETTS INSTITUTE OF TECHNOLOGY                        2 $

## Step 7: Bridge Analysis — Overlap with Existing Graph

Check which DARPA performers already exist in our graph (workshop participants or Coefficient grantees).

In [7]:
# Load existing graph
with open(GRAPH_FILE) as f:
    graph = json.load(f)

existing_nodes = {n['id']: n for n in graph['nodes']}
existing_labels = {n['label'].lower(): n for n in graph['nodes']}

print(f'Existing graph: {len(graph["nodes"])} nodes, {len(graph["edges"])} edges')
print(f'Institution nodes: {sum(1 for n in graph["nodes"] if n["type"] == "institution")}')
print(f'Org nodes: {sum(1 for n in graph["nodes"] if n["type"] == "org")}')

# Alias map for name matching
RECIPIENT_TO_GRAPH = {
    'PRESIDENT AND FELLOWS OF HARVARD COLLEGE': 'inst_Harvard_University',
    'YALE UNIV': 'inst_Yale_University',
    'THE BROAD INSTITUTE, INC': 'inst_Broad_Institute',
    'RUTGERS, THE STATE UNIVERSITY': 'inst_Rutgers_University',
    'UNIVERSITY OF WASHINGTON': 'inst_University_of_Washington',
    'TRUSTEES OF THE UNIVERSITY OF PENNSYLVANIA, THE': 'inst_University_of_Pennsylvania',
    'REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO': None,  # UCSF — not in graph
    'UNIVERSITY OF NORTH CAROLINA AT CHAPEL HILL': None,  # UNC — not in graph
    'THE TRUSTEES OF COLUMBIA UNIVERSITY IN THE CITY OF NEW YORK': 'inst_Columbia_University',
    'GINKGO BIOWORKS, INC.': None,
    'UNIVERSITY OF MASSACHUSETTS MEDICAL SCHOOL': None,
    'LEIDOS, INC.': None,
    'BOOZ ALLEN HAMILTON INC': None,
    'GENERAL ATOMICS': None,
    'DNARX LLC': None,
    'KROMEK LIMITED': None,
}

# Check each recipient against graph
bridges = []
new_orgs = []

for name in recipient_totals:
    # Try explicit mapping first
    if name in RECIPIENT_TO_GRAPH:
        node_id = RECIPIENT_TO_GRAPH[name]
        if node_id and node_id in existing_nodes:
            bridges.append((name, existing_nodes[node_id]['label'], recipient_totals[name]))
            continue
    
    # Try fuzzy match on label
    name_lower = name.lower()
    found = False
    for label_lower, node in existing_labels.items():
        # Simple substring check
        if (name_lower in label_lower or label_lower in name_lower or
            name_lower.replace(',', '').replace('.', '').strip() == label_lower.replace(',', '').replace('.', '').strip()):
            bridges.append((name, node['label'], recipient_totals[name]))
            found = True
            break
    
    if not found:
        new_orgs.append((name, recipient_totals[name]))

print(f'\n=== BRIDGE NODES (already in graph) ===')
print(f"{'USASpending Name':<55} {'Graph Label':<40} {'Amount':>15}")
print('-' * 115)
for usname, glabel, info in sorted(bridges, key=lambda x: x[2]['amount'], reverse=True):
    print(f"{usname[:55]:<55} {glabel[:40]:<40} ${info['amount']:>14,.0f}")

print(f'\n=== NEW ORGANIZATIONS (not in graph) ===')
print(f"{'Recipient':<55} {'Awards':>6} {'Amount':>15}")
print('-' * 80)
for name, info in sorted(new_orgs, key=lambda x: x[1]['amount'], reverse=True):
    print(f"{name[:55]:<55} {info['count']:>6} ${info['amount']:>14,.0f}")

print(f'\nBridge nodes: {len(bridges)}')
print(f'New organizations: {len(new_orgs)}')
bridge_amt = sum(info['amount'] for _, _, info in bridges)
new_amt = sum(info['amount'] for _, info in new_orgs)
print(f'Bridge funding: ${bridge_amt:,.0f}')
print(f'New org funding: ${new_amt:,.0f}')

Existing graph: 502 nodes, 767 edges
Institution nodes: 166
Org nodes: 135

=== BRIDGE NODES (already in graph) ===
USASpending Name                                        Graph Label                                       Amount
-------------------------------------------------------------------------------------------------------------------
REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANCISCO, University of California, San Francisco  $    64,445,623
PRESIDENT AND FELLOWS OF HARVARD COLLEGE                Harvard University                       $    57,210,425
BOOZ ALLEN HAMILTON INC                                 Booz Allen Hamilton                      $    33,084,890
YALE UNIV                                               Yale University                          $    14,323,586
THE BROAD INSTITUTE, INC                                Broad Institute                          $    13,942,324
MASSACHUSETTS INSTITUTE OF TECHNOLOGY                   Massachusetts Institute of Technol

## Step 8: IARPA Fun GCAT Analysis

IARPA awards are not in USASpending.gov (intelligence community funding is largely classified). We searched with broad keywords and found:
- The keyword "functional genomic" returns NIH awards, not IARPA
- IARPA agency filter returns 0 results

**Conclusion**: IARPA Fun GCAT performer data cannot be sourced from USASpending.gov. We'd need press releases or BAA award announcements for this data.

In [8]:
# Verify: IARPA data is not available
with open(os.path.join(RAW_DIR, 'iarpa_all_contracts.json')) as f:
    iarpa = json.load(f)
print(f"IARPA direct query: {len(iarpa.get('results', []))} results")

# The keyword search for "functional genomic" returned NIH, not IARPA
with open(os.path.join(RAW_DIR, 'iarpa_fungcat.json')) as f:
    fungcat = json.load(f)
results = fungcat.get('results', [])
print(f"\n'Functional genomic' keyword search: {len(results)} results")
agencies = Counter(r.get('Awarding Sub Agency', '') for r in results)
print(f'Agencies found: {dict(agencies)}')
print('→ All results are NIH, not IARPA. Fun GCAT data not publicly available via USASpending.')

IARPA direct query: 0 results

'Functional genomic' keyword search: 21 results
Agencies found: {'National Institutes of Health': 10, 'Food and Drug Administration': 2, 'Department of the Air Force': 1, 'Federal Aviation Administration': 4, 'Department of the Army': 1, 'Department of Veterans Affairs': 1, 'Agricultural Research Service': 2}
→ All results are NIH, not IARPA. Fun GCAT data not publicly available via USASpending.


## Step 9: DARPA Program Pages (HTML Snapshots)

We fetched 3 DARPA program pages (PREPARE returned 404). These describe programs generically but don't list specific performers by name.

In [9]:
from html.parser import HTMLParser

class TextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text = []
        self.skip = False
    def handle_starttag(self, tag, attrs):
        if tag in ('script', 'style', 'noscript'):
            self.skip = True
    def handle_endtag(self, tag):
        if tag in ('script', 'style', 'noscript'):
            self.skip = False
    def handle_data(self, data):
        if not self.skip:
            self.text.append(data)
    def get_text(self):
        return re.sub(r'\s+', ' ', ' '.join(self.text)).strip()

html_files = [
    ('P3', 'darpa_page_pandemic-prevention-platform.html'),
    ('PREEMPT', 'darpa_page_preventing-emerging-pathogenic-threats.html'),
    ('SAFE GENES', 'darpa_page_safe-genes.html'),
    ('PREPARE', 'darpa_page_preemptive-expression-protective-alleles-response-elements.html'),
]

for prog_name, fname in html_files:
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        print(f'{prog_name}: file not found')
        continue
    with open(fpath) as f:
        html = f.read()
    
    # Check if 404
    if '404' in html[:500] and 'not found' in html[:500].lower():
        print(f'{prog_name}: 404 Not Found')
        continue
    
    parser = TextExtractor()
    parser.feed(html)
    text = parser.get_text()
    print(f'{prog_name}: {len(text)} chars extracted')
    
    # Check for performer info
    performer_kw = ['performer', 'team', 'university', 'institute', 'company', 'laboratory']
    for kw in performer_kw:
        if kw in text.lower():
            # Extract surrounding context
            matches = re.findall(r'.{0,50}' + re.escape(kw) + r'.{0,50}', text, re.I)
            for m in matches[:2]:
                print(f'  [{kw}] ...{m.strip()}...')
    print()

P3: 5282 chars extracted

PREEMPT: 4467 chars extracted
  [laboratory] ...ent, research is performed entirely in controlled laboratory facilities, including planned proof-of-concept de...

SAFE GENES: 5029 chars extracted
  [performer] ...y progressing field of genome editing. Safe Genes performer teams work across three primary technical focus a...
  [performer] ...al issues related to genome editing technologies. Performer teams will also engage with potential stakeholder...
  [team] ...ing field of genome editing. Safe Genes performer teams work across three primary technical focus areas...
  [team] ...related to genome editing technologies. Performer teams will also engage with potential stakeholders, in...

PREPARE: 2639 chars extracted



## Step 10: Summary Statistics

In [10]:
total_bio_amount = sum(a.get('Award Amount', 0) or 0 for a in bio_awards)
unique_recipients = len(recipient_totals)

print('=' * 60)
print('SPRINT 4 DATA SUMMARY')
print('=' * 60)
print(f'Data source:           USASpending.gov API')
print(f'Agency:                DARPA (Defense Advanced Research Projects Agency)')
print(f'Time range:            2010–2025')
print(f'Raw awards fetched:    {len(all_awards)} (contracts + grants)')
print(f'Bio-relevant awards:   {len(bio_awards)}')
print(f'Excluded (noise):      {len(excluded)}')
print(f'Unique recipients:     {unique_recipients}')
print(f'Total bio funding:     ${total_bio_amount:,.0f}')
print(f'')
print(f'Bridge nodes:          {len(bridges)} (already in graph)')
print(f'New organizations:     {len(new_orgs)}')
print(f'')
print(f'IARPA Fun GCAT:        NOT available in USASpending.gov')
print(f'DARPA program pages:   3 fetched (PREPARE returned 404)')
print(f'                       Pages describe programs, do NOT list specific performers')
print(f'')
print(f'Raw files saved:       data/raw/sprint4/ ({len(os.listdir(RAW_DIR))} files)')

SPRINT 4 DATA SUMMARY
Data source:           USASpending.gov API
Agency:                DARPA (Defense Advanced Research Projects Agency)
Time range:            2010–2025
Raw awards fetched:    110 (contracts + grants)
Bio-relevant awards:   103
Excluded (noise):      7
Unique recipients:     75
Total bio funding:     $472,165,720

Bridge nodes:          22 (already in graph)
New organizations:     53

IARPA Fun GCAT:        NOT available in USASpending.gov
DARPA program pages:   3 fetched (PREPARE returned 404)
                       Pages describe programs, do NOT list specific performers

Raw files saved:       data/raw/sprint4/ (27 files)


## Step 11: Bio+AI Relevance Evaluation

How relevant is this DARPA/IARPA direction to the **biosafety + AI** intersection?

In [11]:
# Check how many awards mention AI/ML/computational methods
AI_KEYWORDS = re.compile(
    r'artificial intelligence|machine learning|deep learning|neural network|'
    r'computational|algorithm|model.*predict|predict.*model|'
    r'data.driven|AI|ML\b|NLP|natural language|'
    r'automated|autonomous|robot',
    re.I
)

ai_awards = [a for a in bio_awards if AI_KEYWORDS.search(a.get('Description', '') or '')]
non_ai_awards = [a for a in bio_awards if not AI_KEYWORDS.search(a.get('Description', '') or '')]

ai_amount = sum(a.get('Award Amount', 0) or 0 for a in ai_awards)
total_amount = sum(a.get('Award Amount', 0) or 0 for a in bio_awards)

print(f'Bio awards with AI/ML mention:    {len(ai_awards)}/{len(bio_awards)} ({100*len(ai_awards)/max(len(bio_awards),1):.0f}%)')
print(f'AI/ML funding:                    ${ai_amount:,.0f} ({100*ai_amount/max(total_amount,1):.0f}% of bio total)')
print(f'Pure bio (no AI mention):         {len(non_ai_awards)} awards, ${total_amount - ai_amount:,.0f}')

if ai_awards:
    print(f'\n--- Awards mentioning AI/ML/computational ---')
    for a in sorted(ai_awards, key=lambda x: x.get('Award Amount', 0) or 0, reverse=True)[:15]:
        amt = a.get('Award Amount', 0) or 0
        print(f'  ${amt:>12,.0f} | {(a.get("Recipient Name") or "")[:40]}')
        print(f'    {(a.get("Description") or "")[:120]}')

print(f'\n=== RELEVANCE ASSESSMENT ===')
print(f'The DARPA BTO portfolio is HIGHLY relevant to biosecurity but has')
print(f'MODERATE overlap with AI specifically. Most awards focus on biological')
print(f'countermeasures, gene editing tools, and pandemic preparedness —')
print(f'the "bio" side of biosafety+AI.')
print(f'')
print(f'Key bridges to the workshop community exist through universities')
print(f'(Harvard, Yale, Columbia, Broad Institute, UPenn, etc.) that have')
print(f'both DARPA BTO funding and NeurIPS BioSafe GenAI workshop authors.')
print(f'')
print(f'RECOMMENDATION: Include DARPA BTO as a funder node with program')
print(f'sub-nodes. Focus on awards to institutions already in the graph')
print(f'(bridge nodes) to keep the graph focused on the bio+AI intersection.')

Bio awards with AI/ML mention:    21/103 (20%)
AI/ML funding:                    $62,218,129 (13% of bio total)
Pure bio (no AI mention):         82 awards, $409,947,591

--- Awards mentioning AI/ML/computational ---
  $  19,590,427 | PRESIDENT AND FELLOWS OF HARVARD COLLEGE
    PROPHECY PROGRAM, ULTRAHIGH THROUGHOUT VIRUS-HOST CELL PICOREACTOR SYSTEM FOR PREDICTIVE MODELING OF VIRAL EVOLUTION.
  $   9,657,579 | UNIVERSITY OF MASSACHUSETTS MEDICAL SCHO
    PROPHECY - PROGRAM TO DEVELOP AN IN VITRO LUNG MODEL AS A CLOSED VIRAL EVOLUTION PLATFORM AND UTILIZE SINGLE VIRION SORT
  $   6,787,986 | CLEMSON UNIVERSITY
    THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFENSE ADVANCED RESEARCH PROJECTS AGENCY (DARPA) BIOLO
  $   3,764,914 | SRI INTERNATIONAL
    THE LIFELONG LEARNING MACHINE (L2M) PROGRAM SEEKS TO DEVELOP NEW MACHINE LEARNING MECHANISMS THAT ENABLE SYSTEMS TO LEAR
  $   3,195,958 | THE LELAND STANFORD JUNIOR UNIVERSITY
    THE LIVING FOUNDRIES ATCG PROGRAM A

## Step 12: Awards Ready for Integration

These are the bio-relevant DARPA awards to institutions that are already in the graph — the highest-confidence additions.

In [12]:
# Show bridge awards — ready for integration
bridge_names = set(name for name, _, _ in bridges)

bridge_awards = [a for a in bio_awards if (a.get('Recipient Name') or '').strip() in bridge_names]

print(f"{'Amount':>15}  {'Recipient':<50} {'Type':<12} Description")
print('=' * 150)
for a in sorted(bridge_awards, key=lambda x: x.get('Award Amount', 0) or 0, reverse=True):
    amt = a.get('Award Amount', 0) or 0
    name = (a.get('Recipient Name') or 'N/A')[:50]
    atype = (a.get('Award Type') or '')[:12]
    desc = (a.get('Description') or 'N/A')[:70]
    print(f'${amt:>14,.0f}  {name:<50} {atype:<12} {desc}')

print(f'\nBridge awards ready for integration: {len(bridge_awards)}')
print(f'Bridge funding total: ${sum(a.get("Award Amount", 0) or 0 for a in bridge_awards):,.0f}')

         Amount  Recipient                                          Type         Description
$    36,151,284  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANC COOPERATIVE  THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFEN
$    33,084,890  BOOZ ALLEN HAMILTON INC                                         BTO BIOMEDICAL ENGINEERING SYSTEMS/MOLECULAR BIOLOGY/BIOSECURITY/MEDIC
$    23,343,160  PRESIDENT AND FELLOWS OF HARVARD COLLEGE           COOPERATIVE  THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFEN
$    19,590,427  PRESIDENT AND FELLOWS OF HARVARD COLLEGE                        PROPHECY PROGRAM, ULTRAHIGH THROUGHOUT VIRUS-HOST CELL PICOREACTOR SYS
$    14,294,461  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANC COOPERATIVE  THE PURPOSE OF THIS AGREEMENT IS TO FUND RESEARCH SUPPORTING THE DEFEN
$    13,999,878  REGENTS OF THE UNIVERSITY OF CALIFORNIA, SAN FRANC              PROPHECY PROGRAM. LINKING VIRUS POPULATION GENETIC STRUCTURE TO IN